# Expérimentations préliminaires — 2. XGBoost

Gradient boosting d'arbres de décision : modèle non linéaire capable de représenter des
interactions entre variables.

Pipeline : imputation (médiane) → standardisation → **ACP (critère de Kaiser, λ > 1)** →
XGBoost, comme pour la régression logistique. Les arbres n'ont besoin ni de standardisation ni
d'ACP pour fonctionner ; l'ACP sert ici uniquement à **alléger le modèle** : elle remplace les
1 052 variables, très redondantes, par un nombre beaucoup plus petit de composantes, ce qui
accélère l'entraînement et réduit le nombre de divisions candidates, donc le risque de
surapprentissage sur du bruit. Pour le choix des hyperparamètres, le prétraitement est ajusté
sur les données d'ajustement seulement ; il est ensuite réajusté sur tout l'entraînement du
pli pour le modèle final. Rien n'est ajusté sur la saison testée.

Réglage volontairement simple (étude préliminaire) :

- taux d'apprentissage 0,03, sous-échantillonnage des matchs et des variables à 80 % ;
- profondeur maximale choisie dans {2, 3, 4} sur la validation interne ;
- nombre d'arbres fixé par arrêt précoce (100 itérations sans amélioration de la log loss de
  validation), puis réajustement sur tout l'entraînement du pli avec ce nombre d'arbres.

**Protocole commun** (voir `src/nhl_forecast/features/equipes.py` et
`src/nhl_forecast/evaluation/preliminaire.py`) :

- saisons régulières 2020-21 à 2025-26, statistiques d'**équipes** seulement, **toutes les
  colonnes** de `t_summary`, `t_percentages`, `t_summaryshooting`, `t_outshootoutshotby` et
  MoneyPuck (5 situations de jeu) ; seuls les identifiants, les colonnes vides ou constantes et
  les doublons exacts sont retirés, automatiquement ;
- validation croisée **chronologique** à 5 plis (fenêtre croissante) : le pli *k* entraîne sur
  les saisons 1 à *k* et teste sur la saison *k* + 1 ;
- hyperparamètres choisis sur les 20 % les plus récents des matchs d'entraînement du pli ;
- métriques : log loss, score de Brier, accuracy (seuil 0,5).

Avec `SAUVER = True`, les tableaux sont enregistrés en CSV (séparateur « ; ») dans
`reports/map6009/tables/` (préfixe `exp_`) et les prédictions dans
`data/processed/experimentation/` (lues par `04_comparaison.ipynb`).

In [1]:
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier

from nhl_forecast.evaluation.preliminaire import (
    GRAINE,
    ACPKaiser,
    exporter_tableau,
    formater,
    plis_chronologiques,
    predictions,
    resultats_par_pli,
    resume,
    sauver_predictions,
    separation_interne,
)
from nhl_forecast.features.equipes import NOM_SAISON, charger_equipes, construire_matchs, variables

SAUVER = True
PROFONDEURS = [2, 3, 4]
PARAMS = dict(objective="binary:logistic", eval_metric="logloss", learning_rate=0.03,
              subsample=0.8, colsample_bytree=0.8, min_child_weight=5, reg_lambda=1.0,
              tree_method="hist", random_state=GRAINE, n_jobs=-1)

In [2]:
matchs = construire_matchs(charger_equipes())
X_cols = variables(matchs)
X, y = matchs[X_cols], matchs["victoire_locale"].to_numpy()
plis = plis_chronologiques(matchs)
print(f"{len(matchs)} matchs, {len(X_cols)} variables")

7428 matchs, 1052 variables


C:\Users\tom_9\Projets\nhl-probabilistic-forecasting\src\nhl_forecast\features\equipes.py:182: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  matchs = matchs.astype({"visiteur": "int64"}).reset_index()


## 1. Entraînement par pli

In [3]:
def pretraitement():
    return make_pipeline(SimpleImputer(strategy="median"), StandardScaler(),
                         ACPKaiser(random_state=GRAINE))  # fmt: skip


preds, choix, acp, importances = [], [], [], []
for pli in plis:
    # (a) profondeur et nombre d'arbres : ACP ajustée sur l'ajustement seulement
    ajust, val = separation_interne(pli["entr"])
    pre = pretraitement().fit(X.iloc[ajust])
    Z_ajust, Z_val = pre.transform(X.iloc[ajust]), pre.transform(X.iloc[val])
    essais = {}
    for d in PROFONDEURS:
        m = XGBClassifier(**PARAMS, max_depth=d, n_estimators=3000, early_stopping_rounds=100)
        m.fit(Z_ajust, y[ajust], eval_set=[(Z_val, y[val])], verbose=False)
        essais[d] = (m.best_score, m.best_iteration + 1)
    d = min(essais, key=lambda k: essais[k][0])
    perte, n_arbres = essais[d]
    # (b) modèle final : ACP réajustée sur tout l'entraînement du pli
    pre = pretraitement().fit(X.iloc[pli["entr"]])
    m = XGBClassifier(**PARAMS, max_depth=d, n_estimators=n_arbres)
    m.fit(pre.transform(X.iloc[pli["entr"]]), y[pli["entr"]])
    p = m.predict_proba(pre.transform(X.iloc[pli["test"]]))[:, 1]
    preds.append(predictions(matchs, pli, p, "XGBoost"))
    saison = NOM_SAISON[pli["saison_test"]]
    choix.append({"Pli": pli["pli"], "Saison testée": saison,
                  "Profondeur": d, "Arbres": n_arbres, "Log loss (validation)": perte})
    acp.append({"Pli": pli["pli"], "Saison testée": saison,
                "Composantes ACP": pre[-1].n_components_,
                "Variance expliquée (%)": round(100 * pre[-1].variance_expliquee_, 1)})
    gain = m.get_booster().get_score(importance_type="gain")  # clés f0, f1, … = CP1, CP2, …
    importances.append(pd.Series({f"CP{int(k[1:]) + 1}": v for k, v in gain.items()},
                                 name=pli["pli"]))
preds = pd.concat(preds, ignore_index=True)
tab_choix = pd.DataFrame(choix)
tab_acp = pd.DataFrame(acp)
formater(tab_choix)

Pli,Saison testée,Profondeur,Arbres,Log loss (validation)
1,2021-22,2,32,0.6721
2,2022-23,3,66,0.6562
3,2023-24,4,91,0.6554
4,2024-25,3,85,0.6617
5,2025-26,4,120,0.6696


In [4]:
formater(tab_acp)

Pli,Saison testée,Composantes ACP,Variance expliquée (%)
1,2021-22,111,95.3000
2,2022-23,133,92.8000
3,2023-24,140,91.8000
4,2024-25,145,91.3000
5,2025-26,149,90.9000


## 2. Résultats par pli

In [5]:
tab_plis = resultats_par_pli(preds)
if SAUVER:
    exporter_tableau(tab_plis, "exp_xgboost_par_pli")
    exporter_tableau(tab_choix, "exp_xgboost_hyperparametres")
    exporter_tableau(tab_acp, "exp_xgboost_composantes")
    sauver_predictions(preds, "02_xgboost")
formater(tab_plis)

Modèle,Pli,Saison testée,n,Log loss,Brier,Accuracy
XGBoost,1,2021-22,1312,0.6749,0.2410,0.5945
XGBoost,2,2022-23,1312,0.6680,0.2376,0.5930
XGBoost,3,2023-24,1312,0.6664,0.2369,0.5998
XGBoost,4,2024-25,1312,0.6739,0.2406,0.5686
XGBoost,5,2025-26,1312,0.6867,0.2468,0.5541


## 3. Résumé (moyenne ± écart-type sur les 5 plis)

In [6]:
tab_resume = resume(preds)
if SAUVER:
    exporter_tableau(tab_resume, "exp_xgboost_resume")
formater(tab_resume)

Modèle,Log loss,Brier,Accuracy
XGBoost,0.6740 ± 0.0080,0.2406 ± 0.0039,0.5820 ± 0.0197


## 4. Importance des composantes

Gain moyen apporté par les divisions sur chaque composante principale, normalisé en % du gain
total de chaque pli, puis moyenné sur les 5 plis (0 si la composante n'existe pas ou n'est
pas utilisée dans un pli). Les composantes sont numérotées par valeur propre décroissante ;
l'ACP étant réajustée à chaque pli, une même composante (p. ex. CP1) n'a pas exactement la
même définition d'un pli à l'autre.

In [7]:
imp = pd.concat(importances, axis=1).fillna(0)
imp = 100 * imp / imp.sum()
tab_imp = (pd.DataFrame({"Gain moyen (%)": imp.mean(axis=1), "Écart-type": imp.std(axis=1)})
           .sort_values("Gain moyen (%)", ascending=False).head(15)
           .rename_axis("Composante").reset_index())
if SAUVER:
    exporter_tableau(tab_imp, "exp_xgboost_importance", 2)
formater(tab_imp, 2)

Composante,Gain moyen (%),Écart-type
CP1,2.55,1.00
CP5,1.40,0.94
CP60,1.26,0.90
CP22,1.25,1.01
CP102,1.25,1.12
CP59,1.22,1.01
CP66,1.19,1.05
CP80,1.18,1.11
CP8,1.18,0.94
CP53,1.16,0.81
